# TalentDesk, Module 3 Section 4 Lab (Exercise): Choosing Mechanisms and Troubleshooting

The capstone. It consolidates the four extension mechanisms of Claude Code, **CLAUDE.md**, **Skills**,
**Subagents**, and **Hooks**, into a decision you can defend, and then repairs a broken setup: a rule
that will not load, a misrouted tool, and an MCP server that will not connect (Lab 2). You classify each
requirement to a mechanism, assemble and validate a **reference architecture**, then run three
diagnose-fix-validate loops. You fill in four short `TODO` blocks; everything else is provided. All four
are testable offline, and live cells wire mechanisms together and confirm the repair. Runs **Sonnet**
(`claude-sonnet-4-6`).

## The real-world scenario

TalentDesk's assistant needs standing conventions, a repeatable offer audit, isolated candidate
discovery, and a hard guardrail that an offer is never sent without a completed background check. Four
needs, four different mechanisms; put a need in the wrong layer and it shows up on your bill or in a
reliability gap.

Then a teammate reports three problems: a screening test rule never applies, a scheduling question keeps
calling the screening tool, and the ATS MCP server fails to connect. Each has a specific, findable cause.
Good troubleshooting is not guessing; it is inspecting what actually loaded, measuring what actually
overlaps, and validating what actually expands.

The question this lab answers: **for each requirement, which mechanism, how do they assemble, and how do
you diagnose and fix the three most common setup failures?**

## Objectives

- Classify each requirement to **CLAUDE.md**, a **Skill**, a **Subagent**, or a **Hook**, with a reason,
  and assemble a validated **reference architecture**.
- Diagnose and fix a **rule that will not load** (a bad glob), a **misrouted tool** (overlapping
  descriptions), and an **MCP connection error** (an unexpanded variable).

## The outcome you should reach

By the end you will have:

- a classifier that routes each requirement to a mechanism, and a validator that confirms the
  architecture has no gaps or misplacements;
- a corrected glob so the screening rule loads, a sharpened description so the scheduling query routes
  correctly, and an MCP config that expands with no missing variables;
- and a single all-green validation over the repaired setup.

Target time: **20 to 30 minutes.** Four small `TODO` blocks, all testable offline. The live cells need a
real key and Node.js 18+.

## How to run

Run top to bottom. The decision engine, the architecture validator, and the troubleshooting loops run
anywhere. The live cells wire mechanisms together and confirm the repair, so paste a real key into
**Setup 2/3** and re-run from the top; **Node.js 18+** is needed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. `pyyaml` parses rule frontmatter; the Agent SDK drives the
live cells and needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q claude-agent-sdk anthropic python-dotenv pyyaml

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live cells.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the sandboxes
import re                                       # glob matching and variable expansion
import sys                                       # detect Windows (special event loop)
import json                                     # read and write configs
import yaml                                      # parse rule frontmatter
import textwrap                                  # keeps the embedded file bodies readable
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cells will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}
    def worker():
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:    box["value"] = loop.run_until_complete(make_coro())
        except Exception as e: box["error"] = e
        finally: loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box: raise box["error"]
    return box.get("value")

print("live model calls:", "ON" if RUN_LIVE else "OFF (decision + troubleshooting cells run offline)")

**This cell:** builds the sandboxes (provided): a `.claude/CLAUDE.md` convention for the live
capstone, and a **broken** setup for Part B: a screening test rule whose glob has a typo, and an
`.mcp.json` whose server URL references a variable that is not set.

In [ ]:
# ===== SETUP 3/3 - build the sandboxes (a convention, and a broken setup) =====
PROJECT = os.path.join(os.getcwd(), "talentdesk_capstone")

def write(rel, content):
    path = os.path.join(PROJECT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

write(".claude/CLAUDE.md", "# TalentDesk conventions\n- Candidate messages are warm and specific.\n")

# --- the broken setup for Part B ---
write(".claude/rules/screening.md", textwrap.dedent("""\
    ---
    paths:
      - "**/tset_*.py"
    ---
    # Screening test rules
    - Mock the ATS; never call it in tests.
    """))                                          # BUG: "tset_" should be "test_"
write("tests/test_screening.py", "# a screening test file\n")
write(".mcp.json", json.dumps({
    "mcpServers": {"ats": {"type": "http", "url": "${ATS_MCP_URL}"}}
}, indent=2))                                      # BUG: the variable is not set anywhere
print("sandbox at", PROJECT)

### The four mechanisms, in one view

- **CLAUDE.md**: always-on context and conventions. Paid for every turn. Best for short, standing rules.
- **Skills**: reusable packaged workflows. Load only when the task matches, so cheap until fired.
- **Subagents**: isolated context and delegated or parallel execution. Heavy fixed overhead per spawn.
- **Hooks**: deterministic interception at lifecycle events. Run outside the model and cannot be
  overridden. Best for guardrails that must run no matter what Claude decides.

Rule of thumb: standing rule → CLAUDE.md; repeatable procedure → Skill; isolation or parallelism →
Subagent; must-happen guardrail → Hook.

---

### 🎯 Part A - route each need, then wire them together

**TODO 1 (about 5 minutes).** Complete `classify()`, the decision engine. Check for a **Hook** need
first (`must`, `never`, `always enforce`, `block`, `guardrail`, `security`), then a **Subagent** need
(`isolate`, `parallel`, `explore`, `without cluttering`, `delegate`), then a **Skill** need (`workflow`,
`procedure`, `checklist`, `reusable`, `on demand`, `audit`); otherwise it is a standing **CLAUDE.md**
convention. Hooks are checked first because "must happen" overrides everything.

In [ ]:
# ===== TODO 1 - classify a requirement to a mechanism =====
HOOK_KW = ["must", "never", "always enforce", "block", "guardrail", "security"]
SUB_KW  = ["isolate", "parallel", "explore", "without cluttering", "delegate"]
SKILL_KW = ["workflow", "procedure", "checklist", "reusable", "on demand", "audit"]

def classify(requirement):                         # requirement text -> (mechanism, reason)
    r = requirement.lower()
    # 👉 TODO 1a: if any HOOK_KW in r -> return ("Hook", "must run deterministically, regardless of model judgment")
    # 👉 TODO 1b: if any SUB_KW  in r -> return ("Subagent", "needs its own context or runs beside other work")
    # 👉 TODO 1c: if any SKILL_KW in r -> return ("Skill", "a repeatable procedure Claude invokes when relevant")
    # 👉 TODO 1d: otherwise -> return ("CLAUDE.md", "a standing convention that applies to most work")
    return ("CLAUDE.md", "default")

REQUIREMENTS = [
    "Never send an offer without a completed background check.",
    "Provide a reusable offer-audit workflow the team can invoke.",
    "Explore the codebase without cluttering the main context.",
    "Use a warm, specific tone in all candidate messages.",
]
for req in REQUIREMENTS:
    mech, why = classify(req)
    print(f"  [{mech:9}] {req}\n             -> {why}")

**Self-check (offline).** Each requirement routes to its intended mechanism.

In [ ]:
# ===== self-check for TODO 1 =====
assert classify("Never send an offer without a completed background check.")[0] == "Hook"
assert classify("Provide a reusable offer-audit workflow the team can invoke.")[0] == "Skill"
assert classify("Explore the codebase without cluttering the main context.")[0] == "Subagent"
assert classify("Use a warm, specific tone in all candidate messages.")[0] == "CLAUDE.md"
print("TODO 1 checks passed")

**This cell:** the mechanism trade-off table (provided): where each lives, when it loads, its token
cost, and whether it is deterministic.

In [ ]:
# ===== the mechanism trade-off table (provided) =====
TABLE = [
    ("CLAUDE.md", "always-on",       "paid every turn",   "model judgment"),
    ("Skill",     "on demand",       "cheap until fired", "model judgment"),
    ("Subagent",  "isolated window", "heavy per spawn",   "model judgment"),
    ("Hook",      "outside context", "zero model tokens", "deterministic"),
]
print(f"  {'mechanism':10} {'loads':16} {'cost':18} determinism")
for name, loads, cost, det in TABLE:
    print(f"  {name:10} {loads:16} {cost:18} {det}")

**TODO 2 (about 4 minutes).** Complete `validate_architecture()`. Given a mapping of requirement to
`(mechanism, artifact)`, return `(gaps, mismatches)` where `gaps` are requirements missing from the map
and `mismatches` are requirements whose chosen mechanism does not equal what `classify()` recommends.

In [ ]:
# ===== TODO 2 - assemble and validate the reference architecture =====
ARCHITECTURE = {                                   # requirement -> (mechanism, concrete artifact)
    "Never send an offer without a completed background check.": ("Hook", "PreToolUse background-check gate"),
    "Provide a reusable offer-audit workflow the team can invoke.": ("Skill", ".claude/skills/offer-audit/SKILL.md"),
    "Explore the codebase without cluttering the main context.": ("Subagent", "Explore subagent (read-only)"),
    "Use a warm, specific tone in all candidate messages.": ("CLAUDE.md", ".claude/CLAUDE.md convention"),
}

def validate_architecture(arch, requirements):    # -> (gaps, mismatches)
    # 👉 TODO 2a: gaps = [req for req in requirements if req not in arch]
    # 👉 TODO 2b: mismatches = [req for req, (mech, _) in arch.items() if classify(req)[0] != mech]
    gaps, mismatches = [], []
    return gaps, mismatches

for req, (mech, artifact) in ARCHITECTURE.items():
    print(f"  {mech:9} <- {artifact}")
gaps, mismatches = validate_architecture(ARCHITECTURE, REQUIREMENTS)
print("\ncoverage gaps:", gaps or "none", "| misplacements:", mismatches or "none")

**Self-check (offline).** The correct architecture has no gaps or mismatches; a deliberately
misplaced guardrail is caught.

In [ ]:
# ===== self-check for TODO 2 =====
gaps, mismatches = validate_architecture(ARCHITECTURE, REQUIREMENTS)
assert gaps == [] and mismatches == [], "the reference architecture should be clean"
broken = dict(ARCHITECTURE)
broken["Never send an offer without a completed background check."] = ("CLAUDE.md", "a note in CLAUDE.md")
assert validate_architecture(broken, REQUIREMENTS)[1], "a guardrail in CLAUDE.md is a misplacement"
print("TODO 2 checks passed")

### Three reference architectures

**1. Single-agent tool-using assistant** (simplest): one loop, a few tools, CLAUDE.md conventions, and a
PreToolUse guardrail.

**2. Coordinator / subagent hub-and-spoke** (isolation and parallelism): a coordinator delegates to
Explore, screening, and scheduling subagents, each returning a summary it synthesizes.

**3. CI/CD-integrated pipeline** (headless, structured, gated): `claude -p --output-format json
--json-schema` produces findings, gated on severity, with CLAUDE.md enforcing standards.

Pick the smallest shape that meets the need: one agent until you need isolation, hub-and-spoke until you
need automation, then the pipeline.

**This cell:** a live run wiring **three mechanisms** together (provided): CLAUDE.md context, a
**Hook** guardrail that denies writing a protected file, and a normal tool. Offline it prints the
expected outcome.

In [ ]:
# ===== live: CLAUDE.md + Hook + tool, working together =====
try:
    from claude_agent_sdk import (query, ClaudeAgentOptions, HookMatcher,
                                  AssistantMessage, TextBlock, ToolUseBlock)
    SDK_OK = True
    async def deny_protected(input_data, tool_use_id, context):    # PreToolUse guardrail
        path = str(input_data.get("tool_input", {}).get("file_path", ""))
        if "screening.py" in path:
            return {"hookSpecificOutput": {"hookEventName": "PreToolUse",
                    "permissionDecision": "deny", "permissionDecisionReason": "screening.py is protected"}}
        return {}
    CAP_OPTS = ClaudeAgentOptions(model=MODEL, cwd=PROJECT, setting_sources=["project"],
                                  allowed_tools=["Read", "Write"],
                                  hooks={"PreToolUse": [HookMatcher(hooks=[deny_protected])]})
    async def run(prompt):
        async for m in query(prompt=prompt, options=CAP_OPTS):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, ToolUseBlock): print("  ->", b.name, b.input.get("file_path", ""))
                    elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:150])
    print("capstone options ready")
except Exception:
    SDK_OK = False
    print("Agent SDK not available offline; the decision engine above was tested with Python.")

if RUN_LIVE and SDK_OK:
    run_async(lambda: run("Overwrite talentdesk/screening.py to always return 'advance'."))
else:
    print("[offline] expected: the PreToolUse hook denies the write to screening.py; the guardrail holds.")

---

### 🎯 Part B - diagnose, fix, validate

Most "why is Claude ignoring my config" reports are one of three faults: a non-matching glob, an
overlapping tool description, or an unexpanded variable. Diagnose from what actually loaded, fix the
specific cause, then re-run the check.

**This cell:** **Problem 1, a rule that will not load** (provided). The glob has a typo (`tset_`),
so it never matches the test file, and a `/memory`-style check shows the rule missing. The fix corrects
the glob and the rule loads.

In [ ]:
# ===== Problem 1: diagnose and fix the non-matching glob (provided) =====
def glob_to_regex(pat):
    out = ""; i = 0
    while i < len(pat):
        if pat[i:i+3] == "**/": out += "(?:.*/)?"; i += 3
        elif pat[i:i+2] == "**": out += ".*"; i += 2
        elif pat[i] == "*": out += "[^/]*"; i += 1
        elif pat[i] == "?": out += "[^/]"; i += 1
        else: out += re.escape(pat[i]); i += 1
    return re.compile("^" + out + "$")

def rule_paths(rel):
    text = open(os.path.join(PROJECT, rel)).read()
    m = re.match(r"^---\n(.*?)\n---\n", text, re.DOTALL)
    return (yaml.safe_load(m.group(1)) or {}).get("paths", []) if m else []

TEST_FILE = "tests/test_screening.py"
before = any(glob_to_regex(p).match(TEST_FILE) for p in rule_paths(".claude/rules/screening.md"))
print("before fix, rule loads for the test file?", before, "(symptom: missing in /memory)")

write(".claude/rules/screening.md", textwrap.dedent("""\
    ---
    paths:
      - "**/test_*.py"
    ---
    # Screening test rules
    - Mock the ATS; never call it in tests.
    """))                                          # fixed: "test_" not "tset_"
after = any(glob_to_regex(p).match(TEST_FILE) for p in rule_paths(".claude/rules/screening.md"))
print("after fix, rule loads for the test file? ", after)

**TODO 3 (about 5 minutes).** Complete `expand()` for **Problem 3, the MCP connection error**. For
each `${VAR}` or `${VAR:-default}`: return the environment value if set; else the default if given; else
leave the literal and record the name in `missing`. An unexpanded variable is a **config** error to fix,
not a transient one to retry.

In [ ]:
# ===== TODO 3 - diagnose the MCP config: expand ${VAR}, collect missing =====
VAR_RE = re.compile(r"\$\{([A-Z_][A-Z0-9_]*)(:-(.*?))?\}")   # ${VAR} or ${VAR:-default}

def expand(value, env):                            # -> (expanded_string, missing_names)
    missing = []
    def repl(m):
        name, _, default = m.group(1), m.group(2), m.group(3)
        # 👉 TODO 3a: if name in env: return env[name]
        # 👉 TODO 3b: if default is not None: return default
        # 👉 TODO 3c: missing.append(name); return m.group(0)   # leave the literal and flag it
        return m.group(0)
    return VAR_RE.sub(repl, value), missing

cfg = json.load(open(os.path.join(PROJECT, ".mcp.json")))
url = cfg["mcpServers"]["ats"]["url"]
expanded, missing = expand(url, dict(os.environ))
category = "config (fix the value); not transient (do not retry)" if missing else "ok"
print("url:", url, "-> expanded:", expanded)
print("missing variables:", missing, "| error category:", category)

**Self-check (offline).** An unset variable is reported missing; a default fills in; setting the
variable resolves it.

In [ ]:
# ===== self-check for TODO 3 =====
assert expand("${ATS_MCP_URL}", {})[1] == ["ATS_MCP_URL"], "unset, no default -> missing"
assert expand("${ATS_MCP_URL:-https://d.example}", {}) == ("https://d.example", []), "default fills in"
fixed_env = dict(os.environ, ATS_MCP_URL="https://mcp.talentdesk.example/ats")
assert expand(url, fixed_env)[1] == [], "with the variable set, nothing is missing"
print("TODO 3 checks passed")

**TODO 4 (about 4 minutes).** Complete `pick_tool()` for **Problem 2, a misrouted tool**. Return the
tool whose description shares the most words with the query. With two identical descriptions a scheduling
query mis-routes to the screening tool; sharpening the description fixes it. The `overlap()` measure is
provided.

In [ ]:
# ===== TODO 4 - diagnose the misrouted tool: pick by description overlap =====
TOOLS = {
    "get_stage":    "Look up candidate information and details.",
    "get_schedule": "Look up candidate information and details.",   # BUG: identical to get_stage
}
def words(s): return set(re.findall(r"[a-z]+", s.lower()))
def overlap(a, b):                                 # Jaccard overlap of two descriptions
    wa, wb = words(a), words(b)
    return len(wa & wb) / len(wa | wb)

def pick_tool(query):                              # -> the tool whose description shares the most words
    qw = words(query)
    # 👉 TODO 4: return max(TOOLS, key=lambda t: len(qw & words(TOOLS[t])))
    return list(TOOLS)[0]

q = "when is the interview scheduled for this candidate"
print("overlap:", round(overlap(TOOLS["get_stage"], TOOLS["get_schedule"]), 2),
      "| picked:", pick_tool(q), "(symptom: misroute)")

TOOLS["get_schedule"] = "Return the interview time and scheduling details for a candidate."   # sharpened
print("overlap:", round(overlap(TOOLS["get_stage"], TOOLS["get_schedule"]), 2),
      "| picked:", pick_tool(q), "(fixed)")

**Self-check (offline).** After sharpening, the scheduling query routes to `get_schedule`.

In [ ]:
# ===== self-check for TODO 4 =====
assert pick_tool("when is the interview scheduled for this candidate") == "get_schedule", "routes correctly after the fix"
assert pick_tool("what stage is this candidate at") in TOOLS, "still returns a valid tool"
print("TODO 4 checks passed")

**This cell:** the whole-setup validation (provided). Re-running all three checks together confirms
the repaired configuration is healthy.

In [ ]:
# ===== validate: all three green (provided) =====
rule_ok = any(glob_to_regex(p).match(TEST_FILE) for p in rule_paths(".claude/rules/screening.md"))
tool_ok = pick_tool("when is the interview scheduled for this candidate") == "get_schedule"
mcp_ok = not expand(url, dict(os.environ, ATS_MCP_URL="https://mcp.talentdesk.example/ats"))[1]
print("rule loads:          ", rule_ok)
print("tool routes correctly:", tool_ok)
print("mcp config expands:   ", mcp_ok)
print("all green:", rule_ok and tool_ok and mcp_ok)

**This cell:** a live confirmation (provided). It points the Agent SDK at the repaired project and
asks about the screening test rule, confirming the now-fixed rule is in scope. Offline it prints the
expected outcome.

In [ ]:
# ===== live: confirm the repaired setup =====
if RUN_LIVE and SDK_OK:
    FIX_OPTS = ClaudeAgentOptions(model=MODEL, cwd=PROJECT, setting_sources=["project"],
                                  allowed_tools=["Read", "Grep", "Glob"])
    async def ask(prompt):
        async for m in query(prompt=prompt, options=FIX_OPTS):
            if isinstance(m, AssistantMessage):
                for b in m.content:
                    if isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])
    run_async(lambda: ask("Open tests/test_screening.py and state the testing rule that applies."))
else:
    print("[offline] expected: the corrected screening test rule is now in scope for the test file.")

---

### Misplacements and symptoms to know

| misplacement | why it fails | right layer |
|---|---|---|
| a security guardrail in CLAUDE.md | advisory only; the model can skip it | Hook |
| a rarely-used procedure in CLAUDE.md | taxes every turn | Skill |
| deep discovery in the main thread | floods the context window | Subagent |
| a standing convention as a skill | may not trigger when needed | CLAUDE.md |

| symptom | likely cause | fix |
|---|---|---|
| a rule never applies | glob does not match the file | correct the `paths` pattern |
| the wrong tool is called | overlapping or vague descriptions | make each description specific |
| an MCP server will not connect | unexpanded `${VAR}` | set the variable; it is a config error |

**Lesson:** the four mechanisms are not interchangeable: CLAUDE.md is what Claude always knows, a
skill is a procedure it runs on demand, a subagent is a clean side-room, and a hook is a rule the harness
enforces. Match each need to the cheapest layer strong enough, assemble from the simplest architecture
that works, and when something breaks, troubleshoot by inspection: check what loaded, measure the
overlap, expand the config, then re-run the check.

---

## Recap - the synthesis and the troubleshooting loop

| Need | Mechanism | Because |
|---|---|---|
| standing convention | CLAUDE.md | always-on, cheap to state |
| repeatable procedure | Skill | loads on demand |
| isolation or parallelism | Subagent | own context window |
| must-happen guardrail | Hook | deterministic, non-overridable |

| Step | Tool | What it tells you |
|---|---|---|
| inspect | `/memory` and a glob check | which rules actually load |
| measure | description overlap | why a tool is misrouted |
| expand | `.mcp.json` variable expansion | why a server will not connect |
| validate | re-run the checks | the fix holds |

**Course complete.** You can now choose the right mechanism, assemble an architecture, and repair a broken
setup. **Try it next:** add a fifth requirement, classify and place it, extend the manifest, then plant a
new fault and run your own diagnose-fix-validate loop.